In [29]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == "notebooks" else Path.cwd().resolve()
RAW_PATH = PROJECT_ROOT / "data/raw/upi_transactions_2024.csv"
PROCESSED_PATH = PROJECT_ROOT / "data/processed/cleaned_upi_transactions_2024.csv"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    columns = (
        df.columns.astype("string")
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    normalized_df = df.copy()
    normalized_df.columns = columns
    return normalized_df


def standardize_text(series: pd.Series) -> pd.Series:
    return (
        series.astype("string")
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
    )


def collect_reasons(index: pd.Index, reason_masks: list[tuple[str, pd.Series]]) -> pd.Series:
    reasons = pd.Series("", index=index, dtype="string")
    for label, mask in reason_masks:
        reasons = reasons.mask(mask & reasons.eq(""), label)
        reasons = reasons.mask(
            mask & ~reasons.eq("") & ~reasons.str.contains(label, regex=False),
            reasons + ";" + label,
        )
    return reasons.str.strip(";")

## Stage 1: Column standardization and type correction
Standardize column names, strip string whitespace, and coerce core fields to the correct data types.

In [30]:
print("=== Stage 1: Column Standardization and Type Correction ===")
raw_df = pd.read_csv(RAW_PATH)
print(f"Rows loaded: {len(raw_df):,}")
print(f"Columns loaded: {len(raw_df.columns)}")
print("Raw columns:")
print(list(raw_df.columns))

clean_df = normalize_columns(raw_df)
print("After column standardization:")
print(list(clean_df.columns))

string_columns = clean_df.select_dtypes(include="object").columns.tolist()
for column in string_columns:
    clean_df[column] = clean_df[column].astype("string").str.strip()
print(f"String columns stripped: {len(string_columns):,}")

if "timestamp" in clean_df.columns:
    clean_df["timestamp"] = pd.to_datetime(clean_df["timestamp"], errors="coerce")
if "amount_inr" in clean_df.columns:
    clean_df["amount_inr"] = pd.to_numeric(clean_df["amount_inr"], errors="coerce")
if "hour_of_day" in clean_df.columns:
    clean_df["hour_of_day"] = pd.to_numeric(clean_df["hour_of_day"], errors="coerce").astype("Int64")
if "is_weekend" in clean_df.columns:
    clean_df["is_weekend"] = pd.to_numeric(clean_df["is_weekend"], errors="coerce").astype("Int64")
if "fraud_flag" in clean_df.columns:
    clean_df["fraud_flag"] = pd.to_numeric(clean_df["fraud_flag"], errors="coerce").astype("Int64")

for column in ["timestamp", "amount_inr", "hour_of_day", "is_weekend", "fraud_flag"]:
    if column in clean_df.columns:
        print(f"{column} dtype: {clean_df[column].dtype}")

=== Stage 1: Column Standardization and Type Correction ===
Rows loaded: 250,000
Columns loaded: 17
Raw columns:
['transaction id', 'timestamp', 'transaction type', 'merchant_category', 'amount (INR)', 'transaction_status', 'sender_age_group', 'receiver_age_group', 'sender_state', 'sender_bank', 'receiver_bank', 'device_type', 'network_type', 'fraud_flag', 'hour_of_day', 'day_of_week', 'is_weekend']
After column standardization:
['transaction_id', 'timestamp', 'transaction_type', 'merchant_category', 'amount_inr', 'transaction_status', 'sender_age_group', 'receiver_age_group', 'sender_state', 'sender_bank', 'receiver_bank', 'device_type', 'network_type', 'fraud_flag', 'hour_of_day', 'day_of_week', 'is_weekend']
String columns stripped: 13
timestamp dtype: datetime64[ns]
amount_inr dtype: int64
hour_of_day dtype: Int64
is_weekend dtype: Int64
fraud_flag dtype: Int64


## Stage 2: Duplicate handling
Remove exact duplicates and then drop duplicate `transaction_id` rows after keeping the first occurrence.

In [31]:
print("=== Stage 2: Duplicate Handling ===")
exact_duplicate_before = len(clean_df)
clean_df = clean_df.drop_duplicates().reset_index(drop=True)
exact_duplicate_removed = exact_duplicate_before - len(clean_df)
print(f"Exact duplicates removed: {exact_duplicate_removed:,}")

if "transaction_id" in clean_df.columns:
    duplicate_transaction_mask = clean_df["transaction_id"].duplicated(keep="first")
    duplicate_transaction_rows = clean_df.loc[duplicate_transaction_mask].copy()
    duplicate_transaction_count = int(duplicate_transaction_mask.sum())
    clean_df = clean_df.loc[~duplicate_transaction_mask].copy().reset_index(drop=True)
else:
    duplicate_transaction_rows = clean_df.iloc[0:0].copy()
    duplicate_transaction_count = 0
print(f"Duplicate transaction_id rows dropped: {duplicate_transaction_count:,}")

=== Stage 2: Duplicate Handling ===
Exact duplicates removed: 0
Duplicate transaction_id rows dropped: 0


## Stage 3: Missing value treatment
Drop rows where critical columns are missing, then fill non-critical categorical gaps with `unknown`.

In [32]:
print("=== Stage 4: Missing Value Treatment ===")
critical_columns = [column for column in ["transaction_id", "timestamp", "amount_inr"] if column in clean_df.columns]
missing_reasons: list[tuple[str, pd.Series]] = []
for column in critical_columns:
    missing_mask = clean_df[column].isna() | clean_df[column].astype("string").str.strip().eq("")
    missing_reasons.append((f"MISSING_CRITICAL_{column.upper()}", missing_mask))
    print(f"{column} missing before drop: {int(missing_mask.sum()):,}")

if missing_reasons:
    missing_combined_mask = pd.concat([mask.rename(label) for label, mask in missing_reasons], axis=1).any(axis=1)
    missing_removed_rows = clean_df.loc[missing_combined_mask].copy()
    missing_removed_rows["reason"] = collect_reasons(clean_df.index, missing_reasons).loc[missing_combined_mask]
    clean_df = clean_df.loc[~missing_combined_mask].copy().reset_index(drop=True)
else:
    missing_removed_rows = clean_df.iloc[0:0].copy()
print(f"Rows dropped for critical missing values: {len(missing_removed_rows):,}")

non_critical_columns = [
    column
    for column in clean_df.columns
    if column not in {"transaction_id", "timestamp", "amount_inr"}
]
for column in non_critical_columns:
    if pd.api.types.is_string_dtype(clean_df[column]) or clean_df[column].dtype == object:
        missing_count = int(clean_df[column].isna().sum())
        clean_df[column] = clean_df[column].astype("string").fillna("unknown")
        if missing_count:
            print(f"{column} filled with 'unknown': {missing_count:,}")

=== Stage 4: Missing Value Treatment ===
transaction_id missing before drop: 0
timestamp missing before drop: 0
amount_inr missing before drop: 0
Rows dropped for critical missing values: 0


## Stage 4: Category standardization
Normalize categorical text, map common variations, and log unexpected values for review.

In [33]:
print("=== Stage 4: Category Standardization ===")
expected_categories = {
    "transaction_type": {"p2p", "p2m", "bill payment", "recharge"},
    "merchant_category": {"food", "grocery", "fuel", "entertainment", "shopping", "healthcare", "education", "transport", "utilities", "other"},
    "transaction_status": {"success", "failed"},
    "sender_age_group": {"18-25", "26-35", "36-45", "46-55", "56+"},
    "receiver_age_group": {"18-25", "26-35", "36-45", "46-55", "56+"},
    "sender_state": {"maharashtra", "karnataka", "delhi", "tamil nadu", "west bengal", "gujarat", "rajasthan", "uttar pradesh", "andhra pradesh", "telangana"},
    "sender_bank": {"sbi", "hdfc", "icici", "axis", "pnb", "kotak", "indusind", "yes bank"},
    "receiver_bank": {"sbi", "hdfc", "icici", "axis", "pnb", "kotak", "indusind", "yes bank"},
    "device_type": {"android", "ios", "web"},
    "network_type": {"3g", "4g", "5g", "wifi"},
    "day_of_week": {"monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"},
}
category_reason_masks: list[tuple[str, pd.Series]] = []
category_mapping = {"wifi": "wifi", "wi-fi": "wifi", "billpayment": "bill payment"}
for column, valid_values in expected_categories.items():
    if column in clean_df.columns:
        normalized_values = standardize_text(clean_df[column]).replace(category_mapping)
        clean_df[column] = normalized_values.fillna("unknown")
        unexpected_mask = normalized_values.notna() & normalized_values.ne("") & ~normalized_values.isin(valid_values)
        category_reason_masks.append((f"UNEXPECTED_{column.upper()}", unexpected_mask))
        print(f"{column} unexpected values flagged: {int(unexpected_mask.sum()):,}")

category_audit_rows = clean_df.loc[
    pd.concat([mask.rename(label) for label, mask in category_reason_masks], axis=1).any(axis=1)
].copy() if category_reason_masks else clean_df.iloc[0:0].copy()
if len(category_audit_rows):
    category_audit_rows["reason"] = collect_reasons(clean_df.index, category_reason_masks).loc[category_audit_rows.index]

=== Stage 4: Category Standardization ===
transaction_type unexpected values flagged: 0
merchant_category unexpected values flagged: 0
transaction_status unexpected values flagged: 0
sender_age_group unexpected values flagged: 0
receiver_age_group unexpected values flagged: 0
sender_state unexpected values flagged: 0
sender_bank unexpected values flagged: 0
receiver_bank unexpected values flagged: 0
device_type unexpected values flagged: 0
network_type unexpected values flagged: 0
day_of_week unexpected values flagged: 0


## Stage 5: Logical consistency checks
Recompute timestamp-derived fields and flag mismatches without hiding them.

In [34]:
print("=== Stage 5: Logical Consistency Checks ===")
consistency_reason_masks: list[tuple[str, pd.Series]] = []
if "timestamp" in clean_df.columns:
    timestamp_parsed = clean_df["timestamp"]
    recomputed_hour = timestamp_parsed.dt.hour.astype("Int64")
    recomputed_day = timestamp_parsed.dt.day_name().str.lower()
    recomputed_weekend = timestamp_parsed.dt.dayofweek.ge(5).astype("Int64")

    if "hour_of_day" in clean_df.columns:
        existing_hour = pd.to_numeric(clean_df["hour_of_day"], errors="coerce").astype("Int64")
        hour_mismatch = timestamp_parsed.notna() & existing_hour.notna() & existing_hour.ne(recomputed_hour)
        consistency_reason_masks.append(("MISMATCH_HOUR_OF_DAY", hour_mismatch))
        clean_df["hour_of_day"] = recomputed_hour.fillna(existing_hour)
        print(f"hour_of_day mismatches corrected/flagged: {int(hour_mismatch.sum()):,}")

    if "day_of_week" in clean_df.columns:
        existing_day = standardize_text(clean_df["day_of_week"]).fillna("unknown")
        day_mismatch = timestamp_parsed.notna() & existing_day.notna() & existing_day.ne(recomputed_day)
        consistency_reason_masks.append(("MISMATCH_DAY_OF_WEEK", day_mismatch))
        clean_df["day_of_week"] = recomputed_day.fillna(existing_day)
        print(f"day_of_week mismatches corrected/flagged: {int(day_mismatch.sum()):,}")

    if "is_weekend" in clean_df.columns:
        existing_weekend = pd.to_numeric(clean_df["is_weekend"], errors="coerce").astype("Int64")
        weekend_mismatch = timestamp_parsed.notna() & existing_weekend.notna() & existing_weekend.ne(recomputed_weekend)
        consistency_reason_masks.append(("MISMATCH_IS_WEEKEND", weekend_mismatch))
        clean_df["is_weekend"] = recomputed_weekend.fillna(existing_weekend).astype("Int64")
        print(f"is_weekend mismatches corrected/flagged: {int(weekend_mismatch.sum()):,}")

if "amount_inr" in clean_df.columns:
    positive_amount_mask = clean_df["amount_inr"].gt(0)
    non_positive_amount_rows = clean_df.loc[~positive_amount_mask].copy()
    print(f"amount_inr non-positive rows flagged: {len(non_positive_amount_rows):,}")
else:
    non_positive_amount_rows = clean_df.iloc[0:0].copy()

binary_rule_issues: dict[str, int] = {}
for column in ["fraud_flag", "is_weekend"]:
    if column in clean_df.columns:
        binary_values = pd.to_numeric(clean_df[column], errors="coerce")
        invalid_binary_mask = ~binary_values.isin([0, 1])
        binary_rule_issues[column] = int(invalid_binary_mask.sum())
        clean_df[column] = binary_values.astype("Int64")
        print(f"{column} invalid binary values flagged: {binary_rule_issues[column]:,}")

=== Stage 5: Logical Consistency Checks ===
hour_of_day mismatches corrected/flagged: 0
day_of_week mismatches corrected/flagged: 0
is_weekend mismatches corrected/flagged: 0
amount_inr non-positive rows flagged: 0
fraud_flag invalid binary values flagged: 0
is_weekend invalid binary values flagged: 0


## Stage 7: Outlier detection
Outliers are identified with the IQR rule and flagged in `is_outlier_amount`. They are kept in the dataset; this stage only records them for review.

In [35]:
print("=== Stage 6: Outlier Detection (Not Removed) ===")
if "amount_inr" in clean_df.columns:
    amount_numeric = pd.to_numeric(clean_df["amount_inr"], errors="coerce")
    q1 = amount_numeric.quantile(0.25)
    q3 = amount_numeric.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    clean_df["is_outlier_amount"] = amount_numeric.lt(lower_bound) | amount_numeric.gt(upper_bound)
    outlier_rows_df = clean_df.loc[clean_df["is_outlier_amount"]].copy()
    print(f"Q1: {q1:.2f} | Q3: {q3:.2f} | IQR: {iqr:.2f}")
    print(f"IQR lower bound: {lower_bound:.2f}")
    print(f"IQR upper bound: {upper_bound:.2f}")
    print(f"Outlier rows flagged: {len(outlier_rows_df):,}")
else:
    clean_df["is_outlier_amount"] = False
    outlier_rows_df = clean_df.iloc[0:0].copy()

=== Stage 6: Outlier Detection (Not Removed) ===
Q1: 288.00 | Q3: 1596.00 | IQR: 1308.00
IQR lower bound: -1674.00
IQR upper bound: 3558.00
Outlier rows flagged: 21,171


In [36]:
print("=== Audit Summary ===")
print(f"Rows before processing: {len(raw_df):,}")
print(f"Rows after duplicate removal: {len(clean_df):,}")
print(f"Rows dropped for missing critical values: {len(missing_removed_rows):,}")
print(f"Duplicate transaction_id rows dropped: {duplicate_transaction_count:,}")
print(f"Suspicious rows flagged: {len(suspicious_rows_df):,}")
print(f"Outlier rows flagged (kept): {len(outlier_rows_df):,}")
if len(clean_df):
    print(f"Retention after required drops: {(len(clean_df) / len(raw_df)) * 100:.2f}%")

print("\nSuspicious sample:")
if len(suspicious_rows_df):
    suspicious_rows_df.head(10)
else:
    print("No suspicious rows detected.")

PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
clean_df.to_csv(PROCESSED_PATH, index=False)

print("\n=== Export Complete ===")
print(f"Saved cleaned dataset to: {PROCESSED_PATH}")
print(f"Final rows written: {len(clean_df):,}")
print(f"Suspicious rows flagged: {len(suspicious_rows_df):,}")
print(f"Outlier rows flagged and retained: {len(outlier_rows_df):,}")

=== Audit Summary ===
Rows before processing: 250,000
Rows after duplicate removal: 250,000
Rows dropped for missing critical values: 0
Duplicate transaction_id rows dropped: 0
Suspicious rows flagged: 0
Outlier rows flagged (kept): 21,171
Retention after required drops: 100.00%

Suspicious sample:
No suspicious rows detected.

=== Export Complete ===
Saved cleaned dataset to: /Users/suryanshchattree/Desktop/DVA_CapStone/UPI_2024/nst-dva-capstone-2-project/data/processed/cleaned_upi_transactions_2024.csv
Final rows written: 250,000
Suspicious rows flagged: 0
Outlier rows flagged and retained: 21,171
